# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/aidanzanefrondozo/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df[feature_names].values
y = df["Class"].values

In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/20
13/13 [==============================] - 0s 8ms/step - loss: 95.1425 - accuracy: 0.3030 - val_loss: 48.5991 - val_accuracy: 0.4800
Epoch 2/20
13/13 [==============================] - 0s 2ms/step - loss: 32.0415 - accuracy: 0.4545 - val_loss: 8.2492 - val_accuracy: 0.2400
Epoch 3/20
13/13 [==============================] - 0s 2ms/step - loss: 12.9716 - accuracy: 0.3131 - val_loss: 9.1078 - val_accuracy: 0.1600
Epoch 4/20
13/13 [==============================] - 0s 2ms/step - loss: 7.3109 - accuracy: 0.2323 - val_loss: 4.6711 - val_accuracy: 0.2000
Epoch 5/20
13/13 [==============================] - 0s 2ms/step - loss: 5.7150 - accuracy: 0.2121 - val_loss: 2.7040 - val_accuracy: 0.4800
Epoch 6/20
13/13 [==============================] - 0s 2ms/step - loss: 2.3727 - accuracy: 0.3636 - val_loss: 1.6983 - val_accuracy: 0.4800
Epoch 7/20
13/13 [==============================] - 0s 2ms/step - loss: 1.5884 - accuracy: 0.4545 - val_loss: 1.6709 - val_accuracy: 0.4400
Epoch 8/20
13/13

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 0.7963
2/2 [==============================] - 0s 997us/step

Classification Report:
              precision    recall  f1-score   support

     class_0       0.76      1.00      0.86        19
     class_1       0.92      0.57      0.71        21
     class_2       0.75      0.86      0.80        14

    accuracy                           0.80        54
   macro avg       0.81      0.81      0.79        54
weighted avg       0.82      0.80      0.79        54

Confusion Matrix:
[[19  0  0]
 [ 5 12  4]
 [ 1  1 12]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
import os

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

file_size_kb = os.path.getsize("model_base.tflite") / 1024
print(f"Base TFLite model size: {file_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpxihm4f_i/assets


INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpxihm4f_i/assets
2026-05-20 16:04:48.893237: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.


Base TFLite model size: 14.07 KB


2026-05-20 16:04:48.893254: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 16:04:48.893691: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpxihm4f_i
2026-05-20 16:04:48.894319: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 16:04:48.894326: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpxihm4f_i
2026-05-20 16:04:48.896027: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 16:04:48.896679: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 16:04:48.927480: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpxihm4f_i

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]

def file_size_kb(filename):
    return os.path.getsize(filename) / 1024

def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    y_pred = []
    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)

        # Quantize input if needed
        if input_details['dtype'] == np.int8:
            scale, zero_point = input_details['quantization']
            sample = (sample / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])

        # Dequantize output if needed
        if output_details['dtype'] == np.int8:
            scale, zero_point = output_details['quantization']
            output = (output.astype(np.float32) - zero_point) * scale

        y_pred.append(np.argmax(output))
    
    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)
    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")
    print(f"\nClassification Report ({quant_type.upper()}):")
    print(classification_report(y_true, y_pred, target_names=wine.target_names))
    print(f"Confusion Matrix ({quant_type.upper()}):")
    print(confusion_matrix(y_true, y_pred))

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='int8',     filename='model_int8.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='float16',  filename='model_float16.tflite')
quantize_and_evaluate(model, X_test, y_test_cat, quant_type='dynamic',  filename='model_dynamic.tflite')

INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpk5voiq3e/assets


INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpk5voiq3e/assets
/Users/aidanzanefrondozo/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 5.74 KB

INT8 TFLite model size: 5.74 KB

Classification Report (INT8):
              precision    recall  f1-score   support

     class_0       0.86      0.95      0.90        19
     class_1       0.82      0.86      0.84        21
     class_2       0.82      0.64      0.72        14

    accuracy                           0.83        54
   macro avg       0.83      0.82      0.82        54
weighted avg       0.83      0.83      0.83        54

Confusion Matrix (INT8):
[[18  1  0]
 [ 1 18  2]
 [ 2  3  9]]
INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp9xc_9gj_/assets


2026-05-20 16:04:58.741719: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 16:04:58.741737: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 16:04:58.741913: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpk5voiq3e
2026-05-20 16:04:58.742556: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 16:04:58.742562: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpk5voiq3e
2026-05-20 16:04:58.744378: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 16:04:58.773262: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpk5voiq3e
2026-05-


FLOAT16 TFLite model size: 8.95 KB

FLOAT16 TFLite model size: 8.95 KB

Classification Report (FLOAT16):
              precision    recall  f1-score   support

     class_0       0.76      1.00      0.86        19
     class_1       0.92      0.52      0.67        21
     class_2       0.71      0.86      0.77        14

    accuracy                           0.78        54
   macro avg       0.79      0.79      0.77        54
weighted avg       0.81      0.78      0.76        54

Confusion Matrix (FLOAT16):
[[19  0  0]
 [ 5 11  5]
 [ 1  1 12]]
INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp3nq5r2rl/assets


2026-05-20 16:04:59.091603: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 16:04:59.091616: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 16:04:59.091740: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp9xc_9gj_
2026-05-20 16:04:59.092383: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 16:04:59.092388: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp9xc_9gj_
2026-05-20 16:04:59.094181: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 16:04:59.122801: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp9xc_9gj_
2026-05-


DYNAMIC TFLite model size: 8.17 KB

DYNAMIC TFLite model size: 8.17 KB

Classification Report (DYNAMIC):
              precision    recall  f1-score   support

     class_0       0.73      1.00      0.84        19
     class_1       0.91      0.48      0.62        21
     class_2       0.71      0.86      0.77        14

    accuracy                           0.76        54
   macro avg       0.78      0.78      0.75        54
weighted avg       0.79      0.76      0.74        54

Confusion Matrix (DYNAMIC):
[[19  0  0]
 [ 6 10  5]
 [ 1  1 12]]


2026-05-20 16:04:59.417610: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 16:04:59.417623: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 16:04:59.417745: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp3nq5r2rl
2026-05-20 16:04:59.418341: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 16:04:59.418347: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp3nq5r2rl
2026-05-20 16:04:59.420018: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 16:04:59.447739: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmp3nq5r2rl
2026-05-

## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
batch_size = 8
epochs = 20
dataset_size = len(X_train)
end_step = int(np.ceil(dataset_size / batch_size)) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(Dense(64, activation='relu', input_shape=(num_features,)), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(32, activation='relu'), pruning_schedule=pruning_schedule),
    prune_low_magnitude(Dense(num_classes, activation='softmax'), pruning_schedule=pruning_schedule)
])

In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_history = pruned_model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks
)

Epoch 1/10
13/13 [==============================] - 1s 9ms/step - loss: 12.0985 - accuracy: 0.4444 - val_loss: 5.0023 - val_accuracy: 0.4400
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 4.2247 - accuracy: 0.4444 - val_loss: 1.2925 - val_accuracy: 0.5600
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 2.3167 - accuracy: 0.5152 - val_loss: 2.8710 - val_accuracy: 0.5200
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 1.6815 - accuracy: 0.6061 - val_loss: 1.4251 - val_accuracy: 0.6400
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 1.4154 - accuracy: 0.5859 - val_loss: 1.0277 - val_accuracy: 0.6400
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 1.0609 - accuracy: 0.6162 - val_loss: 1.1644 - val_accuracy: 0.5600
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 1.1744 - accuracy: 0.6061 - val_loss: 1.7380 - val_accuracy: 0.6000
Epoch 8/10
13/13 [=

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned_model)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpqi9egx6g/assets


INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpqi9egx6g/assets


Pruned TFLite model size: 14.14 KB


2026-05-20 16:05:12.989508: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 16:05:12.989524: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 16:05:12.989667: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpqi9egx6g
2026-05-20 16:05:12.990097: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 16:05:12.990102: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpqi9egx6g
2026-05-20 16:05:12.991392: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 16:05:13.003616: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpqi9egx6g
2026-05-

In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_pruned = np.argmax(stripped_model.predict(X_test), axis=1)

print("\nClassification Report (Pruned):")
print(classification_report(y_test, y_pred_pruned, target_names=wine.target_names))

print("Confusion Matrix (Pruned):")
print(confusion_matrix(y_test, y_pred_pruned))

2/2 [==============================] - 0s 1ms/step

Classification Report (Pruned):
              precision    recall  f1-score   support

     class_0       0.86      0.95      0.90        19
     class_1       0.91      0.48      0.62        21
     class_2       0.50      0.79      0.61        14

    accuracy                           0.72        54
   macro avg       0.76      0.74      0.71        54
weighted avg       0.78      0.72      0.72        54

Confusion Matrix (Pruned):
[[18  0  1]
 [ 1 10 10]
 [ 2  1 11]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_soft_labels = model.predict(X_train)

4/4 [==============================] - 0s 738us/step


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
y_train_combined = np.concatenate([y_train_cat, teacher_soft_labels], axis=1)

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss

In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

student_history = student_model.fit(
    X_train, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/10
13/13 [==============================] - 0s 8ms/step - loss: 37.7477 - accuracy: 0.3434 - val_loss: 14.7255 - val_accuracy: 0.2800
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 9.0174 - accuracy: 0.3838 - val_loss: 7.0880 - val_accuracy: 0.6400
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 4.2564 - accuracy: 0.6061 - val_loss: 2.5751 - val_accuracy: 0.4400
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 2.6016 - accuracy: 0.4646 - val_loss: 2.4515 - val_accuracy: 0.6000
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 1.5665 - accuracy: 0.5455 - val_loss: 1.5741 - val_accuracy: 0.6000
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 1.5815 - accuracy: 0.5051 - val_loss: 1.3044 - val_accuracy: 0.6000
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 1.3186 - accuracy: 0.4949 - val_loss: 1.3801 - val_accuracy: 0.6800
Epoch 8/10
13/13 [

In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd_model)

print(f"Knowledge Distillation TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpu9ftj51q/assets


INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpu9ftj51q/assets


Knowledge Distillation TFLite model size: 6.10 KB


2026-05-20 16:05:27.796839: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 16:05:27.796853: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 16:05:27.796977: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpu9ftj51q
2026-05-20 16:05:27.797578: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 16:05:27.797583: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpu9ftj51q
2026-05-20 16:05:27.799439: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 16:05:27.827681: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpu9ftj51q
2026-05-

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_student = np.argmax(student_model.predict(X_test), axis=1)

print("\nClassification Report (Knowledge Distillation):")
print(classification_report(y_test, y_pred_student, target_names=wine.target_names))

print("Confusion Matrix (Knowledge Distillation):")
print(confusion_matrix(y_test, y_pred_student))

2/2 [==============================] - 0s 1ms/step

Classification Report (Knowledge Distillation):
              precision    recall  f1-score   support

     class_0       1.00      0.84      0.91        19
     class_1       0.56      0.90      0.69        21
     class_2       0.00      0.00      0.00        14

    accuracy                           0.65        54
   macro avg       0.52      0.58      0.54        54
weighted avg       0.57      0.65      0.59        54

Confusion Matrix (Knowledge Distillation):
[[16  1  2]
 [ 0 19  2]
 [ 0 14  0]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# <-- (if needed) Enter your code here <--#
# ── 1. Define pruning schedule for the smaller student architecture ──
dataset_size_kd = len(X_train)
end_step_kd = int(np.ceil(dataset_size_kd / 8)) * 20

pruning_schedule_kd = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.8,   # push sparsity higher since the base model is already small
    begin_step=0,
    end_step=end_step_kd
)

# ── 2. Build pruned student model ──
pruned_student = Sequential([
    prune_low_magnitude(Dense(32, activation='relu', input_shape=(num_features,)), pruning_schedule=pruning_schedule_kd),
    prune_low_magnitude(Dense(16, activation='relu'), pruning_schedule=pruning_schedule_kd),
    prune_low_magnitude(Dense(num_classes, activation='softmax'), pruning_schedule=pruning_schedule_kd)
])

# ── 3. Train with distillation loss + pruning callback ──
pruned_student.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

pruned_student.fit(
    X_train, y_train_combined,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()]
)

# ── 4. Strip pruning wrappers ──
stripped_student = tfmot.sparsity.keras.strip_pruning(pruned_student)

# ── 5. Apply int8 quantization on top ──
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_student)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_combined = converter.convert()
with open("model_combined.tflite", "wb") as f:
    f.write(tflite_combined)

print(f"Combined model size: {file_size_kb('model_combined.tflite'):.2f} KB")

# ── 6. Evaluate via TFLite interpreter ──
quantize_and_evaluate(
    stripped_student, X_test, y_test_cat,
    quant_type='int8',
    filename='model_combined.tflite'
)

# ── 7. Size summary across all parts ──
print("\n── Model Size Comparison ──")
for name, path in [
    ("Base",             "model_base.tflite"),
    ("Int8",             "model_int8.tflite"),
    ("Float16",          "model_float16.tflite"),
    ("Dynamic",          "model_dynamic.tflite"),
    ("Pruned",           "model_pruned.tflite"),
    ("KD Student",       "model_kd.tflite"),
    ("Combined (final)", "model_combined.tflite"),
]:
    print(f"  {name:<20} {file_size_kb(path):.2f} KB")

Epoch 1/20
13/13 [==============================] - 1s 11ms/step - loss: 14.2910 - accuracy: 0.4646 - val_loss: 4.4510 - val_accuracy: 0.4800
Epoch 2/20
13/13 [==============================] - 0s 2ms/step - loss: 2.7748 - accuracy: 0.5253 - val_loss: 2.5786 - val_accuracy: 0.6400
Epoch 3/20
13/13 [==============================] - 0s 2ms/step - loss: 1.8898 - accuracy: 0.6162 - val_loss: 1.5919 - val_accuracy: 0.5200
Epoch 4/20
13/13 [==============================] - 0s 2ms/step - loss: 1.4832 - accuracy: 0.6263 - val_loss: 1.5400 - val_accuracy: 0.5600
Epoch 5/20
13/13 [==============================] - 0s 2ms/step - loss: 1.5742 - accuracy: 0.5253 - val_loss: 1.5235 - val_accuracy: 0.5200
Epoch 6/20
13/13 [==============================] - 0s 2ms/step - loss: 1.1901 - accuracy: 0.6768 - val_loss: 2.7810 - val_accuracy: 0.4400
Epoch 7/20
13/13 [==============================] - 0s 2ms/step - loss: 1.8415 - accuracy: 0.6061 - val_loss: 1.4904 - val_accuracy: 0.5600
Epoch 8/20
13/13 [

INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmps0h__7js/assets
/Users/aidanzanefrondozo/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 17:38:54.377047: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 17:38:54.377566: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 17:38:54.377941: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmps0h__7js
2026-05-20 17:38:54.378403: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 17:38:54.378410: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_

Combined model size: 3.70 KB
INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpj5plsp4n/assets


 20628 microseconds.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
INFO:tensorflow:Assets written to: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpj5plsp4n/assets
/Users/aidanzanefrondozo/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 3.70 KB

INT8 TFLite model size: 3.70 KB

Classification Report (INT8):
              precision    recall  f1-score   support

     class_0       0.00      0.00      0.00        19
     class_1       0.00      0.00      0.00        21
     class_2       0.26      1.00      0.41        14

    accuracy                           0.26        54
   macro avg       0.09      0.33      0.14        54
weighted avg       0.07      0.26      0.11        54

Confusion Matrix (INT8):
[[ 0  0 19]
 [ 0  0 21]
 [ 0  0 14]]

── Model Size Comparison ──
  Base                 14.07 KB
  Int8                 5.74 KB
  Float16              8.95 KB
  Dynamic              8.17 KB
  Pruned               14.14 KB
  KD Student           6.10 KB
  Combined (final)     3.70 KB


2026-05-20 17:38:54.692612: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 17:38:54.692624: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 17:38:54.692782: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpj5plsp4n
2026-05-20 17:38:54.693235: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 17:38:54.693242: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpj5plsp4n
2026-05-20 17:38:54.694411: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 17:38:54.706150: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/zf/m3_1trv10w192w5h1_st7k4c0000gn/T/tmpj5plsp4n
2026-05-

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
